# UniMol v2 — PXR pEC50 Fine-tuning (aug2, 8-Fold, Kaggle)

**Training data**: `train_unimol_aug_set1_holdout30_dedup.csv`  
- clean_train2 + crude_nv_hi + semi_nv + **223 Set 1 unblinded labels** = **4,262 compounds**  
- HA ≥ 5.5: **502** compounds | HA ≥ 6.0: **112** compounds  
- Holdout-30 **excluded** from training (used for honest RAE validation)  
- No sc_inactives supplement (confirmed to hurt UniMol potent-tail predictions)

**Two inference targets**:  
1. `set1_holdout_30_stratified.csv` — 30-compound holdout for honest RAE validation  
2. `test.csv` — 513-compound Set 2 for final submission blend

**No isotonic calibration** — raw predictions used for blending (calibration confirmed to hurt RAE)

**Required Kaggle dataset files**:  
- `train_unimol_aug_set1_holdout30_dedup.csv`  
- `set1_holdout_30_stratified.csv`  
- `test.csv`

## Cell 1 — GPU Check

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    raise SystemExit('No GPU detected — enable GPU accelerator in Kaggle settings.')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q unimol-tools huggingface_hub

from unimol_tools import MolTrain, MolPredict
from rdkit import Chem
print('unimol-tools + rdkit ready')

## Cell 3 — Configuration

In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# ── Column names ──────────────────────────────────────────────────────────────
SMILES_COL = 'SMILES'
TARGET_COL = 'pEC50'
NAME_COL   = 'Molecule Name'

# ── Locate input files ────────────────────────────────────────────────────────
def find_file(filename):
    for root, dirs, files in os.walk('/kaggle/input'):
        if filename in files:
            return os.path.join(root, filename)
    raise FileNotFoundError(f'{filename} not found under /kaggle/input')

TRAIN_CSV   = find_file('train_unimol_aug_set1_holdout30_dedup.csv')
HOLDOUT_CSV = find_file('set1_holdout_30_stratified.csv')
TEST_CSV    = find_file('test.csv')
print(f'Train   : {TRAIN_CSV}')
print(f'Holdout : {HOLDOUT_CSV}')
print(f'Test    : {TEST_CSV}')

# ── UniMol hyperparameters ────────────────────────────────────────────────────
UNIMOL_MODEL      = 'unimolv2'
UNIMOL_SIZE       = '84m'      # '164m' if VRAM > 16 GB
UNIMOL_EPOCHS     = 40
UNIMOL_BATCH_SIZE = 4
UNIMOL_LR         = 1e-4
UNIMOL_EARLY_STOP = 10
UNIMOL_KFOLD      = 8
UNIMOL_SPLIT      = 'scaffold'

# ── Conformer generation ──────────────────────────────────────────────────────
CONF_SEED         = 42
CONF_MAX_ATTEMPTS = 5
REMOVE_HS         = False

# ── Output paths ──────────────────────────────────────────────────────────────
UNIMOL_SAVE_DIR       = '/kaggle/working/unimol_pxr_aug2_8fold'
HOLDOUT_OUT_CSV       = '/kaggle/working/predictions_unimol_aug2_holdout30_raw.csv'
TEST_OUT_RAW_CSV      = '/kaggle/working/predictions_unimol_aug2_test_raw.csv'
OOF_CSV               = '/kaggle/working/oof_unimol_aug2.csv'

GLOBAL_SEED = 42
print(f'\nUniMol {UNIMOL_MODEL} ({UNIMOL_SIZE}) — {UNIMOL_KFOLD}-fold scaffold CV')
print(f'Training : train_unimol_aug_set1_holdout30_dedup.csv (4262 rows)')
print(f'Epochs: {UNIMOL_EPOCHS}  Batch: {UNIMOL_BATCH_SIZE}  LR: {UNIMOL_LR}')
print(f'Inference targets: holdout-30 (validation) + test.csv (submission)')

## Cell 4 — Load & Validate Data

In [ ]:
def load_and_validate(path, require_target=True):
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    # normalise column names
    for col in [SMILES_COL, TARGET_COL, NAME_COL]:
        if col not in df.columns:
            match = next((c for c in df.columns if c.lower() == col.lower()), None)
            if match:
                df.rename(columns={match: col}, inplace=True)
    if SMILES_COL not in df.columns:
        raise ValueError(f"'{SMILES_COL}' column not found in {path}. Columns: {list(df.columns)}")
    if require_target and TARGET_COL not in df.columns:
        raise ValueError(f"'{TARGET_COL}' column not found in {path}.")
    before = len(df)
    cols_to_check = [SMILES_COL, TARGET_COL] if require_target else [SMILES_COL]
    df = df.dropna(subset=cols_to_check).reset_index(drop=True)
    if len(df) < before:
        print(f'  Dropped {before - len(df)} rows with missing values from {os.path.basename(path)}')
    return df

train_df   = load_and_validate(TRAIN_CSV,   require_target=True)
holdout_df = load_and_validate(HOLDOUT_CSV, require_target=True)
test_df    = load_and_validate(TEST_CSV,    require_target=False)

y_train    = train_df[TARGET_COL].values.astype(np.float32)
y_holdout  = holdout_df[TARGET_COL].values.astype(np.float32)

def rae(y, yhat):
    return float(np.sum(np.abs(y - yhat)) / np.sum(np.abs(y - y.mean())))

print(f'Train   : N={len(train_df)}  pEC50=[{y_train.min():.2f}, {y_train.max():.2f}]  '
      f'mean={y_train.mean():.3f}')
print(f'          HA>=5.5: {(y_train>=5.5).sum()}  HA>=6.0: {(y_train>=6.0).sum()}  '
      f'Inact<4.0: {(y_train<4.0).sum()}')
print(f'Holdout : N={len(holdout_df)}  pEC50=[{y_holdout.min():.2f}, {y_holdout.max():.2f}]')
print(f'Test    : N={len(test_df)}')

## Cell 5 — Generate 3D Conformers

Conformers generated for **train**, **holdout-30**, and **test** using ETKDG v3 + MMFF94.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem

def generate_conformers(smiles_list, seed=42, max_attempts=5, remove_hs=False, label=''):
    atoms_list, coords_list, failed = [], [], []
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            atoms_list.append(None); coords_list.append(None); failed.append(i)
            continue
        mol_h = Chem.AddHs(mol)
        ok = -1
        for attempt in range(max_attempts):
            p = AllChem.ETKDGv3()
            p.randomSeed = seed + attempt
            p.numThreads = 0
            p.enforceChirality = True
            p.useSmallRingTorsions = True
            ok = AllChem.EmbedMolecule(mol_h, p)
            if ok == 0:
                break
        if ok != 0:
            ok = AllChem.EmbedMolecule(mol_h, AllChem.ETKDG())
        if ok != 0:
            atoms_list.append(None); coords_list.append(None); failed.append(i)
            continue
        try:
            AllChem.MMFFOptimizeMolecule(mol_h, maxIters=2000)
        except Exception:
            pass
        if remove_hs:
            mol_h = Chem.RemoveHs(mol_h)
        atoms_list.append([a.GetSymbol() for a in mol_h.GetAtoms()])
        coords_list.append(mol_h.GetConformer().GetPositions().astype(np.float64))
    print(f'{label}: {len(smiles_list)} molecules  |  failed: {len(failed)}')
    return atoms_list, coords_list, failed

t0 = time.time()
train_atoms,   train_coords,   train_failed   = generate_conformers(
    train_df[SMILES_COL].tolist(),   seed=CONF_SEED, max_attempts=CONF_MAX_ATTEMPTS,
    remove_hs=REMOVE_HS, label='Train')
holdout_atoms, holdout_coords, holdout_failed = generate_conformers(
    holdout_df[SMILES_COL].tolist(), seed=CONF_SEED, max_attempts=CONF_MAX_ATTEMPTS,
    remove_hs=REMOVE_HS, label='Holdout')
test_atoms,    test_coords,    test_failed    = generate_conformers(
    test_df[SMILES_COL].tolist(),    seed=CONF_SEED, max_attempts=CONF_MAX_ATTEMPTS,
    remove_hs=REMOVE_HS, label='Test')
print(f'Conformer generation complete in {(time.time()-t0)/60:.1f} min')

## Cell 6 — 8-Fold UniMol Fine-tuning

In [ ]:
from unimol_tools import MolTrain

os.makedirs(UNIMOL_SAVE_DIR, exist_ok=True)

valid_mask   = [a is not None for a in train_atoms]
n_failed     = sum(1 for v in valid_mask if not v)
n_valid      = sum(valid_mask)

if n_failed > 0:
    print(f'WARNING: {n_failed} train conformers failed — filtering to {n_valid} valid.')
    filt_smiles  = [s for s, v in zip(train_df[SMILES_COL].tolist(), valid_mask) if v]
    filt_atoms   = [a for a, v in zip(train_atoms,  valid_mask) if v]
    filt_coords  = [c for c, v in zip(train_coords, valid_mask) if v]
    filt_targets = [t for t, v in zip(train_df[TARGET_COL].tolist(), valid_mask) if v]
else:
    print(f'All {n_valid} train conformers generated successfully.')
    filt_smiles  = train_df[SMILES_COL].tolist()
    filt_atoms   = train_atoms
    filt_coords  = train_coords
    filt_targets = train_df[TARGET_COL].tolist()

train_data = {
    'SMILES'      : filt_smiles,
    'atoms'       : filt_atoms,
    'coordinates' : filt_coords,
    TARGET_COL    : filt_targets,
}

print('=' * 65)
print(f'UniMol {UNIMOL_MODEL} ({UNIMOL_SIZE}) — {UNIMOL_KFOLD}-Fold Fine-tuning')
print(f'Train samples : {len(filt_smiles)}')
print(f'Epochs        : up to {UNIMOL_EPOCHS}  early_stop={UNIMOL_EARLY_STOP}')
print(f'Batch / LR    : {UNIMOL_BATCH_SIZE} / {UNIMOL_LR}')
print('=' * 65)

t0 = time.time()
clf = MolTrain(
    task           = 'regression',
    data_type      = 'molecule',
    epochs         = UNIMOL_EPOCHS,
    learning_rate  = UNIMOL_LR,
    batch_size     = UNIMOL_BATCH_SIZE,
    early_stopping = UNIMOL_EARLY_STOP,
    metrics        = 'mae',
    split          = UNIMOL_SPLIT,
    kfold          = UNIMOL_KFOLD,
    save_path      = UNIMOL_SAVE_DIR,
    remove_hs      = REMOVE_HS,
    smiles_col     = 'SMILES',
    target_cols    = TARGET_COL,
    model_name     = UNIMOL_MODEL,
    model_size     = UNIMOL_SIZE,
    use_cuda       = True,
    seed           = GLOBAL_SEED,
)
fit_result = clf.fit(data=train_data)
t_elapsed  = (time.time() - t0) / 60
print(f'\nFine-tuning complete in {t_elapsed:.1f} min')
model_files = sorted(glob.glob(os.path.join(UNIMOL_SAVE_DIR, 'model_*.pth')))
print(f'Saved models: {[os.path.basename(m) for m in model_files]}')

## Cell 7 — OOF Evaluation

Per-fold OOF predictions on training data. No isotonic calibration — raw predictions used for blending.

In [ ]:
import shutil, tempfile, re
from unimol_tools import MolPredict
from rdkit.Chem.Scaffolds import MurckoScaffold
from collections import defaultdict

def scaffold_kfold_indices(smiles_list, n_folds, seed=42):
    scaffold_to_idx = defaultdict(list)
    for i, smi in enumerate(smiles_list):
        try:
            mol  = Chem.MolFromSmiles(smi)
            scaf = MurckoScaffold.MurckoScaffoldSmiles(
                mol=mol, includeChirality=False) if mol else smi
        except:
            scaf = smi
        scaffold_to_idx[scaf].append(i)
    groups     = sorted(scaffold_to_idx.values(), key=len, reverse=True)
    folds      = [[] for _ in range(n_folds)]
    fold_sizes = [0] * n_folds
    for group in groups:
        smallest = min(range(n_folds), key=lambda x: fold_sizes[x])
        folds[smallest].extend(group)
        fold_sizes[smallest] += len(group)
    return [np.array(f) for f in folds]

filt_y           = np.array(filt_targets, dtype=np.float32)
fold_val_indices = scaffold_kfold_indices(filt_smiles, n_folds=UNIMOL_KFOLD, seed=GLOBAL_SEED)
y_oof            = np.full(len(filt_y), np.nan)
config_src       = os.path.join(UNIMOL_SAVE_DIR, 'config.yaml')
support_files    = ['target_scaler.ss', 'metric.result']

print('Running per-fold OOF predictions...')
for fold_i, (model_path, val_idx) in enumerate(zip(model_files, fold_val_indices)):
    tmp_dir = tempfile.mkdtemp(prefix=f'fold{fold_i}_')
    try:
        shutil.copy(model_path, os.path.join(tmp_dir, 'model_0.pth'))
        for sf in support_files:
            src = os.path.join(UNIMOL_SAVE_DIR, sf)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(tmp_dir, sf))
        with open(config_src) as f:
            cfg = f.read()
        for key in ['kfold', 'fold_num', 'n_model', 'num_fold']:
            cfg = re.sub(rf'{key}\s*:\s*\d+', f'{key}: 1', cfg)
        with open(os.path.join(tmp_dir, 'config.yaml'), 'w') as f:
            f.write(cfg)
        val_smi    = [filt_smiles[i] for i in val_idx]
        val_df_tmp = pd.DataFrame({'SMILES': val_smi, TARGET_COL: 0.0})
        pred_fold  = MolPredict(load_model=tmp_dir)
        fold_preds = np.array(pred_fold.predict(data=val_df_tmp)).flatten()
        y_oof[val_idx] = fold_preds
        fold_mae = float(np.mean(np.abs(filt_y[val_idx] - fold_preds)))
        fold_sp  = float(spearmanr(filt_y[val_idx], fold_preds).statistic)
        print(f'  fold {fold_i}: N={len(val_idx)}  MAE={fold_mae:.4f}  Spearman={fold_sp:.4f}')
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

missing = np.isnan(y_oof)
if missing.any():
    print(f'  WARNING: {missing.sum()} OOF missing — filling with mean')
    y_oof[missing] = filt_y.mean()

oof_rae = rae(filt_y, y_oof)
oof_mae = float(np.mean(np.abs(filt_y - y_oof)))
oof_sp  = float(spearmanr(filt_y, y_oof).statistic)
print(f'\nOOF  RAE={oof_rae:.4f}  MAE={oof_mae:.4f}  Spearman={oof_sp:.4f}')
print(f'OOF pred range : [{y_oof.min():.3f}, {y_oof.max():.3f}]')
print(f'True     range : [{filt_y.min():.3f}, {filt_y.max():.3f}]')
print(f'Compression    : {y_oof.std()/filt_y.std():.3f}  (1.0 = no compression)')

## Cell 8 — Predict: Holdout-30 (Validation)

Holdout-30 was excluded from training. RAE computed against true pEC50 values for honest model evaluation.

In [ ]:
from unimol_tools import MolPredict

def predict_on(df, atoms, coords, label):
    """Run full-ensemble prediction. Returns raw_preds array aligned to df rows."""
    n = len(df)
    data = {
        'SMILES'      : df[SMILES_COL].tolist(),
        'atoms'       : atoms,
        'coordinates' : coords,
        TARGET_COL    : [0.0] * n,
    }
    print(f'Predicting {n} {label} compounds...')
    t0 = time.time()
    predictor = MolPredict(load_model=UNIMOL_SAVE_DIR)
    raw       = np.array(predictor.predict(data=data)).flatten()
    # Guard: pad if MolPredict dropped failed conformers
    if len(raw) < n:
        print(f'  WARNING: got {len(raw)} preds for {n} compounds — padding with training mean')
        preds     = np.full(n, float(np.nanmean(raw)))
        valid_idx = [i for i, a in enumerate(atoms) if a is not None]
        for k, i in enumerate(valid_idx[:len(raw)]):
            preds[i] = raw[k]
    else:
        preds = raw
    print(f'  Done in {(time.time()-t0)/60:.1f} min  '
          f'range=[{preds.min():.3f}, {preds.max():.3f}]  '
          f'mean={preds.mean():.3f}  std={preds.std():.3f}')
    return preds

holdout_preds = predict_on(holdout_df, holdout_atoms, holdout_coords, 'holdout-30')

# ── Evaluate against true pEC50 ───────────────────────────────────────────────
holdout_rae = rae(y_holdout, holdout_preds)
holdout_mae = float(np.mean(np.abs(y_holdout - holdout_preds)))
holdout_sp  = float(spearmanr(y_holdout, holdout_preds).statistic)
holdout_bias = float((holdout_preds - y_holdout).mean())
print(f'\nHoldout-30 Results (honest out-of-sample):')
print(f'  RAE      = {holdout_rae:.4f}')
print(f'  MAE      = {holdout_mae:.4f}')
print(f'  Spearman = {holdout_sp:.4f}')
print(f'  Bias     = {holdout_bias:+.4f}')
print()
# Per-bin
for lo, hi, label in [(-99,4,'<4 Inactive'),(4,5.5,'4-5.5 Moderate'),(5.5,99,'>5.5 Potent')]:
    mask = (y_holdout > lo) & (y_holdout <= hi)
    if mask.any():
        b = float((holdout_preds - y_holdout)[mask].mean())
        m = float(np.abs(holdout_preds - y_holdout)[mask].mean())
        print(f'  {label}: n={mask.sum():2d}  bias={b:+.3f}  MAE={m:.3f}')

## Cell 9 — Predict: Test Set (Submission)

In [ ]:
test_preds = predict_on(test_df, test_atoms, test_coords, 'test (Set 2)')

## Cell 10 — Save All Outputs

Three files saved to `/kaggle/working/`:
- `predictions_unimol_aug2_holdout30_raw.csv` — holdout-30 predictions + true values (for RAE verification)
- `predictions_unimol_aug2_test_raw.csv` — test Set 2 raw predictions (for ensemble blending)
- `oof_unimol_aug2.csv` — OOF predictions (for blend weight optimisation)

In [ ]:
# ── Holdout-30: predictions + true values for RAE verification ────────────────
holdout_out = holdout_df[[NAME_COL, SMILES_COL, TARGET_COL]].copy()
holdout_out['pEC50_pred'] = holdout_preds
holdout_out['error']      = holdout_preds - holdout_out[TARGET_COL]
holdout_out.to_csv(HOLDOUT_OUT_CSV, index=False)
print(f'Holdout-30 predictions: {HOLDOUT_OUT_CSV}')
print(f'  RAE={holdout_rae:.4f}  MAE={holdout_mae:.4f}  Spearman={holdout_sp:.4f}')
print(f'  range=[{holdout_preds.min():.3f}, {holdout_preds.max():.3f}]  N={len(holdout_out)}')

print()

# ── Test Set 2: raw predictions for blending ──────────────────────────────────
name_col_test = next((c for c in test_df.columns
                      if c.lower().replace(' ', '_') in ('molecule_name', 'id')), None)
test_out = pd.DataFrame()
if name_col_test:
    test_out[NAME_COL] = test_df[name_col_test].values
else:
    test_out[NAME_COL] = [f'compound_{i+1}' for i in range(len(test_df))]
test_out[SMILES_COL] = test_df[SMILES_COL].values
test_out['pEC50']    = test_preds
test_out.to_csv(TEST_OUT_RAW_CSV, index=False)
print(f'Test Set 2 predictions: {TEST_OUT_RAW_CSV}')
print(f'  range=[{test_preds.min():.3f}, {test_preds.max():.3f}]  '
      f'mean={test_preds.mean():.3f}  std={test_preds.std():.3f}  N={len(test_out)}')
print(f'  >5.5: {(test_preds>5.5).sum()}  >6.0: {(test_preds>6.0).sum()}')

print()

# ── OOF: for blend weight optimisation ───────────────────────────────────────
oof_out = pd.DataFrame({
    NAME_COL      : train_df.loc[train_df[SMILES_COL].isin(filt_smiles), NAME_COL].values
                    if NAME_COL in train_df.columns else [f'train_{i}' for i in range(len(filt_smiles))],
    'SMILES'      : filt_smiles,
    'pEC50_true'  : filt_targets,
    'pEC50_pred'  : y_oof,
})
oof_out.to_csv(OOF_CSV, index=False)
print(f'OOF predictions: {OOF_CSV}  N={len(oof_out)}')
print(f'  OOF RAE={oof_rae:.4f}  MAE={oof_mae:.4f}  Spearman={oof_sp:.4f}')

print(f'\n{"="*60}')
print('SUMMARY')
print(f'  Training    : {len(filt_smiles)} compounds (aug2 clean)')
print(f'  OOF         : RAE={oof_rae:.4f}  MAE={oof_mae:.4f}')
print(f'  Holdout-30  : RAE={holdout_rae:.4f}  MAE={holdout_mae:.4f}  (honest)')
print(f'  Test range  : [{test_preds.min():.3f}, {test_preds.max():.3f}]')
print(f'\nFiles in /kaggle/working/:')
print(f'  {os.path.basename(HOLDOUT_OUT_CSV)}')
print(f'  {os.path.basename(TEST_OUT_RAW_CSV)}')
print(f'  {os.path.basename(OOF_CSV)}')
print(f'{"="*60}')